Question: How nonlinear do predictors have to be? Answer: not very nonlinear!
- Linear predictor type on rank-20 network performs almost as well as softplus.

But this is surprising. It would appear from the first few PCs of network
activity that the states are nonlinearly tangled.

There must be L.I. dimensions that separate the task and network states
relatively well. Goal of this notebook is to visualize those dimensions.

In [1]:
%load_ext autoreload
%autoreload 2 --print

In [13]:
import torch as th
import torch.jit as jit
import copy
import numpy as np
import dill
from collections import namedtuple
from torch import optim
from torch import nn
from dynrn.rnntasks import (
    DriscollTasks,
    DriscollPlots,
    period_start_mask,
    periwindows,
    periperiod_sliced,
    split_trials,
    extract_trial_data,
    apply_to_trial_groups,
    nanmean_with_req,
    split_trials_driscoll,
)
from dynrn.predictors import (
    activity_dataset,
    save_dsn,
    load_dsn,
    discounted_sums,
    MultiBlockActivityDataset,
    periodwise_pca
)
import dynrn.predictors as pred
import dynrn.basic_rnns as rnns
from dynrn.basic_rnns import find_hash
from sklearn.cluster import KMeans
from dynrn.viz import dynamics as vd
import scipy.stats
from cmap import Colormap
from scipy.stats import uniform, norm
from datetime import datetime
from mplutil import util as vu
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.decomposition import PCA
import tqdm
from numpy import linalg as la
import os
import joblib as jl
import time
import seaborn as sns

In [3]:
from dynrn.viz import styles
from dynrn.viz.styles import getc
t20 = lambda x: getc(f"seaborn:tab20{x}")
colors, plotter = styles.init_plt(
    Path('../plots/notebook/rnn-ortho').resolve(),
    fmt = 'pdf', display=False)
plot_root = Path(plotter.plot_dir)

period_colors = {
    "iti": t20("b:5"),
    "context": t20("c:1"),
    "stim": t20("b:2"),
    "memory": t20("b:14"),
    "response": t20("b:10")
}

In [4]:
# cuda setup
device = th.device('cuda' if th.cuda.is_available() else 'cpu')
cpu = th.device('cpu' if th.cuda.is_available() else 'cpu')
print(device.type)

cpu


### Load predictions

In [7]:
root_dir = "/Users/kaifox/projects/loop/dynrn/data"

# Load activity dataset
act_data_hash = '7d4c36'
act_path = find_hash(root_dir, act_data_hash, ".dil")
act_data = dill.load(open(act_path, "rb"))
print("loaded avtivity data:", Path(act_path).name)

# Select main variables from activity dataset
test_data: MultiBlockActivityDataset = act_data["test"]
test_blocks = DriscollTasks.split_dataset(test_data)

# set plotting directory
act_name = Path(act_path).name
nb_plot_dir = Path('../plots/notebook').resolve()
plotter.plot_dir = nb_plot_dir / 'rnn-ortho' / act_name
plotter.plot_dir.mkdir(exist_ok=True)

loaded avtivity data: mems-smoo-long-lr20_7d4c36.dil


### Activity data PCA and clustering



In [11]:
period_pca, pc_act, _, fpc_act, _ = periodwise_pca(test_data['periods'], test_data['activity'], n_dim = 2)

In [12]:
clust = KMeans(n_clusters=50).fit_predict(fpc_act.reshape(-1, 2))

(150, 500, 2, 5)